In [1]:
from datasets import load_dataset
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer,TrainingArguments,Trainer
from peft import LoraConfig, get_peft_model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\VICTUS\OneDrive\Desktop\Generative AI\genAI-playground\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
model_name = "bert-base-uncased"
dataset = load_dataset("stanfordnlp/imdb")
tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\VICTUS\OneDrive\Desktop\Generative AI\genAI-playground\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VICTUS\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\VICTUS\OneDrive\Desktop\Generative AI\genAI-playground\venv\Lib\site-pa

In [3]:
def preprocess(example):
    return tokenizer(example["text"],truncation= True, padding="max_length",max_length=128)
encoded_dataset = dataset.map(preprocess,batched=True)
encoded_dataset = encoded_dataset.rename_column("label","labels")
encoded_dataset.set_format("torch")

Map: 100%|██████████| 50000/50000 [00:07<00:00, 6275.21 examples/s]


In [9]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels = 2)
model = get_peft_model(model, lora_config)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 735.58it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

KeyError: None

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-imdb",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    num_train_epochs=2,
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"].shuffle(seed=42).select(range(2000)),
    eval_dataset=encoded_dataset["test"].shuffle(seed=42).select(range(1000)),
)

trainer.train()

model.save_pretrained("./lora-imdb-adapter")

In [ ]:
from transformers import pipeline

sentiment = pipeline(
    "text-classification",
    model="./lora-imdb-adapter",
    tokenizer="bert-base-uncased"
)

label_map = {
    "LABEL_0": "NEGATIVE",
    "LABEL_1": "POSITIVE"
}

examples = [
    "I absolutely loved this film—best sci-fi I've seen in years!",
    "It was okay, not great, but worth a watch.",
    "Terrible plot, terrible acting, total waste of time."
]

for text in examples:
    result = sentiment(text)[0]
    print(
        f"{text[:40]}... → "
        f"{label_map[result['label']]} "
        f"({result['score']:.2f})"
    )